# Sesión 1 · Herramientas y el bucle

**Prácticas de *LLMs aplicados a Finanzas* · MIAX · jueves 10 de septiembre**

## Antes de empezar: por qué no vamos a montar un RAG

Lo esperable, en un curso de LLMs sobre documentos, sería montar un RAG
clásico: trocear los informes, calcular sus *embeddings*, guardarlos en un
índice vectorial y, ante cada pregunta, recuperar los *k* fragmentos más
parecidos y metérselos al modelo en el prompt. Recuperar, y luego generar. Es
el patrón que describe el paper de 2020 que le puso nombre
([Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP
Tasks*](https://arxiv.org/abs/2005.11401)) y sigue siendo el punto de partida
razonable para casi cualquier sistema de este tipo.

Vamos a construir todas esas piezas. Lo que no vamos a hacer es dejar que sean
**la arquitectura**, y conviene decir por qué.

Un RAG clásico decide **de antemano** qué información necesita el modelo. La
recuperación ocurre siempre, una sola vez, antes de generar, con la pregunta
tal y como llegó y con una *k* fijada de antemano. Eso funciona mientras la
pregunta se parezca a un párrafo del corpus. Deja de funcionar en cuanto:

- **la respuesta no está en un sitio, sino en dos.** «¿Qué riesgos añadió
  Microsoft entre FY2024 y FY2025?» exige recuperar dos veces y comparar. Una
  sola pasada de recuperación devuelve fragmentos de los dos años mezclados y
  el modelo se los inventa a medias;
- **el dato exacto no está en la prosa, sino en una tabla estructurada.**
  Buscar «beneficio neto de Apple» por similitud semántica devuelve párrafos
  que *hablan* del beneficio neto. La cifra auditada está en el XBRL, y no
  hace falta buscarla: se consulta;
- **la pregunta se refiere a algo que no existe.** Un RAG plano siempre
  devuelve sus *k* fragmentos, aunque la compañía por la que preguntáis no
  esté en el corpus. Siempre recupera algo, y ese algo siempre parece una
  respuesta.

La alternativa no es tirar el retrieval: es **bajarlo de arquitectura a
herramienta**. En lugar de un *pipeline* fijo que recupera y luego genera,
un modelo con varias herramientas que decide sobre la marcha cuál usar,
cuántas veces y con qué consulta. La documentación de LangChain llama a lo
primero *2-Step RAG* y a lo segundo *Agentic RAG*, y compara los dos en
[docs.langchain.com/oss/python/langchain/retrieval](https://docs.langchain.com/oss/python/langchain/retrieval).

Y el retrieval sobrevive ahí dentro por dos razones que no tienen nada que ver
con lo que el modelo sea capaz de leer:

- **Coste.** El corpus son 649.119 tokens. Meterlo entero en cada pregunta es
  pagarlo entero en cada pregunta. Lo calculáis vosotros en §2.
- **Auditabilidad.** En un entorno regulado hay que poder señalar el párrafo
  del que sale la cifra. El contexto largo da la respuesta; el retrieval da el
  ancla.

*(RAG lo visteis en teoría el sábado 12. Esto es lo que se hace con ello.)*

## Qué construimos hoy

Un **agente investigador sobre informes 10-K de la SEC**. Al final de la
sesión tendrá cuatro herramientas y sabrá elegir entre ellas:

| Herramienta | Para qué |
| --- | --- |
| `list_available()` | Comprobar qué hay en el corpus antes de inventárselo |
| `get_xbrl_fact()` | La cifra exacta, tal y como la reportó la compañía |
| `search_filings()` | Buscar en el texto de los informes. Hoy es caja negra |
| `read_section()` | El texto completo de una sección. Cara |

Esas cuatro herramientas son las tres carencias de arriba, resueltas. El
agente puede **recuperar dos veces** y comparar, porque quien decide cuántas
búsquedas hacen falta es él y no el *pipeline*. Puede **no buscar**, y
consultar la tabla XBRL cuando lo que se le pide es una cifra exacta. Y puede
**comprobar que algo existe** antes de responder, en lugar de devolver los
cinco fragmentos más parecidos a una pregunta sobre una empresa que no está.

El precio de esa flexibilidad es que el sistema se vuelve menos predecible: un
*pipeline* fijo siempre hace lo mismo, y un agente no. Por eso la sesión que
viene va entera de medirlo —evaluación de trayectoria, no solo de respuesta— y
de ponerle límites. Hoy toca que funcione; el día 17, que aguante.

Fijaos en la asimetría que hay en la tabla, porque es el eje de todo el curso:
`get_xbrl_fact` es barata, exacta y determinista, y `search_filings` es cara,
difusa y aproximada. **Elegir bien entre las dos es el trabajo del agente**, y
es lo que se evalúa. Un agente que acierta la cifra leyéndola de la prosa está
mal aunque el número salga bien: la próxima vez, con otra tabla partida, saldrá
mal y nadie se enterará.

## Qué se entrega el 24

Repositorio, golden set de 20 preguntas vuestras con al menos 6 comparativas,
informe con la tabla *baseline* contra final, y una presentación de 8 minutos
en la que ejecutáis **10 preguntas ciegas** que no veis hasta ese día. Todo
está en el enunciado que tenéis en la mano.

## Un aviso sobre el orden del curso

Hoy vais a escribir un bucle ReAct y a recuperar texto de un corpus. **El
viernes 18 os explicarán ReAct y el paper; RAG lo visteis el sábado 12.** Es
deliberado: hoy construís, y la teoría llega después a ponerle nombre a algo
que ya habréis tocado con las manos.

## Cómo se usa este notebook

- **Se ejecuta de arriba abajo, sin saltos.** No hay estado oculto: si saltáis
  una celda, la siguiente falla.
- Cada sección termina con `assert`. Si pasan, podéis seguir.
- **Ninguna clave está escrita aquí dentro.** Se piden por `getpass`.
- Lo que puede fallar por red está en `try/except` y degrada a un modo sin esa
  función. Nada de esto debería parar la clase.

Junto al notebook necesitáis dos ficheros más: `miax_s1.py` y
`demo_traza.json`. Y los dos ZIP del corpus, que se montan en §1.


In [ ]:
# Preparación automática de los archivos en Google Colab, antes de la demo.
# En cada entorno nuevo se descarga el repositorio público. Si se vuelve a
# ejecutar esta celda en el mismo entorno, se reutiliza la copia existente.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
except ImportError:
    print("Ejecución local: se utilizan los archivos de la carpeta actual.")
else:
    URL_REPO = "https://github.com/rodovilllapa210/https---github.com-PRACTICA-AGENTE-RAG.git"
    CARPETA_REPO = Path("/content/MIAX_2026/Practica_Agente_RAG")

    if not (CARPETA_REPO / ".git").is_dir():
        CARPETA_REPO.parent.mkdir(parents=True, exist_ok=True)
        try:
            subprocess.run(
                ["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
                 URL_REPO, str(CARPETA_REPO)],
                check=True,
            )
        except (OSError, subprocess.CalledProcessError) as e:
            raise RuntimeError(
                "No se pudo descargar el repositorio público de GitHub. "
                "Comprueba la conexión y la disponibilidad del repositorio."
            ) from e

    NECESARIOS = (
        "miax_s1.py", "demo_traza.json", "corpus_miax_2026.zip",
        "indice_faiss.zip", "golden_set_ejemplo.jsonl",
    )
    faltan = [nombre for nombre in NECESARIOS
              if not (CARPETA_REPO / nombre).is_file()]
    if faltan:
        raise FileNotFoundError(
            "Faltan archivos en el repositorio descargado: " + ", ".join(faltan)
        )

    os.chdir(CARPETA_REPO)
    if str(CARPETA_REPO) not in sys.path:
        sys.path.insert(0, str(CARPETA_REPO))
    print("Archivos del proyecto preparados en", CARPETA_REPO)


In [ ]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Tarda alrededor de minuto y medio: mientras corre, leed la celda siguiente.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-google-genai==4.3.7 google-genai==2.10.0 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2 google-auth==2.49.0
print("Instalación terminada.")


## Por qué van fijadas las versiones

Porque LangChain publica cada pocos días y una API que se mueve debajo de un
notebook lo rompe sin tocar una línea de código.

No es una precaución teórica: **los notebooks de la edición anterior de este
curso ya no ejecutan.** `create_react_agent`, `MemorySaver`,
`with_structured_output` y `RunnableWithMessageHistory` eran la API correcta
hace un año y hoy son otra cosa. Lo que veis aquí está verificado contra la
documentación oficial el 2 de septiembre de 2026.

De ahí salen dos hábitos que valen para cualquier proyecto que dependa de un
proveedor de modelos:

1. **Fijad la versión exacta**, no el rango. `>=1.3` no es una versión.
2. **Anotad la fecha de verificación** al lado del pin, para saber cómo de
   viejo es lo que estáis leyendo.


In [ ]:
# Clave de Gemini. Se recupera de los secretos de Google Colab y se deja
# en el entorno de esta sesión, sin mostrar su valor ni pedirlo por teclado.
import os
from google.colab import userdata

HAY_CLAVE = False
try:
    clave = userdata.get("GEMINI_API_KEY")
    if clave:
        os.environ["GEMINI_API_KEY"] = clave
        HAY_CLAVE = True
        print("GEMINI_API_KEY: recuperada de los secretos de Colab.")
    else:
        print("GEMINI_API_KEY: el secreto está vacío. Revisa los secretos de Colab.")
except Exception:
    print("No se pudo recuperar el secreto GEMINI_API_KEY de Colab. "
          "Comprueba que existe y que este notebook tiene permiso para acceder a él.")

if not HAY_CLAVE:
    print("Sin clave: las celdas que llaman al modelo no van a funcionar, "
          "pero las de datos sí.")

# Una sola clave. La observabilidad de este curso no necesita ninguna más:
# la trayectoria se imprime en el propio notebook (§6).


In [ ]:
# %% Corpus e indice  --------------------------------
# Los dos ZIP os los pasamos nosotros (Drive compartido, aula virtual o el
# panel de ficheros de Colab): son 5,6 MB entre los dos. Nada de descargar
# de EDGAR en vivo, que con treinta cuadernos a la vez acaba en bloqueo.
#
# Si los teneis en Drive:
#     from google.colab import drive; drive.mount("/content/drive")
# y anadid la carpeta a CANDIDATOS.
import hashlib, pathlib, zipfile

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
URL_RESPALDO = ""          # vacio si no estan alojados
DESTINO = pathlib.Path("corpus")

CANDIDATOS = [
    pathlib.Path("."),
    pathlib.Path("/content"),
    pathlib.Path("/content/drive/MyDrive/MIAX_2026"),
    pathlib.Path("/content/drive/Shareddrives/MIAX_2026"),
]


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


def _localizar(nombre):
    for base in CANDIDATOS:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    if URL_RESPALDO:
        import urllib.request
        destino = pathlib.Path(nombre)
        urllib.request.urlretrieve(f"{URL_RESPALDO}/{nombre}", destino)
        return destino
    return None


try:
    for nombre, esperado in PAQUETES:
        origen = _localizar(nombre)
        assert origen is not None, (
            f"No encuentro {nombre}. Subelo con el panel de ficheros de "
            f"Colab (icono de carpeta a la izquierda), o monta el Drive "
            f"donde este. Buscado en: {[str(c) for c in CANDIDATOS]}"
        )
        obtenido = _sha256(origen)
        assert obtenido == esperado, (
            f"{nombre} no coincide con lo esperado: el fichero esta "
            f"corrupto o es de otra version.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}"
        )
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)

    # Los dos manifiestos declaran el hash de chunks.jsonl. El indice se
    # construyo sobre ESE fichero: si no cuadra, el indice y sus metadatos
    # estan desalineados y el retrieval devuelve el texto equivocado sin
    # dar ningun error.
    huella = _sha256(DESTINO / "chunks.jsonl")
    for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
        ruta = DESTINO / manifiesto
        if ruta.exists():
            assert huella in ruta.read_text(encoding="utf-8"), (
                f"chunks.jsonl no cuadra con {manifiesto}: el indice se "
                "construyo sobre otros fragmentos."
            )

    print("Corpus e indice verificados en", DESTINO.resolve())
    for p in sorted(DESTINO.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(DESTINO))
            print(f"  {rel:28s} {p.stat().st_size / 1e6:7.2f} MB")

except Exception as e:
    print("No se pudo preparar el corpus:", e)
    print("Pide los ficheros al profesor y dejalos junto al notebook.")


In [ ]:
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.8-flash"

modelo = None
if HAY_CLAVE:
    try:
        modelo = init_chat_model(MODELO, temperature=0)
        print("Modelo preparado:", MODELO)
    except Exception as e:
        print(f"No se pudo crear el modelo ({type(e).__name__}: {e}).")

# temperature=0 en todo lo evaluable para que baseline y sistema final
# sean comparables.
import pathlib
assert pathlib.Path("corpus/chunks.jsonl").is_file(), \
    "El corpus no está preparado."
assert pathlib.Path("corpus/indice/corpus.faiss").is_file(), \
    "Falta el índice FAISS."
print("§1 listo.")


## Anatomía de un 10-K

El 10-K es el informe anual que toda empresa cotizada en EE. UU. presenta ante
la SEC. Es un documento normalizado: los mismos epígrafes, en el mismo orden,
todos los años y en todas las compañías. Eso es lo que lo hace utilizable como
corpus.

Nos quedamos con cuatro epígrafes, que son donde está lo que se puede
preguntar:

| Item | Qué contiene | Qué se le pregunta |
| --- | --- | --- |
| **1A** · Risk Factors | Los riesgos que la compañía declara | Qué riesgos nuevos aparecen, cómo cambian entre ejercicios |
| **7** · MD&A | La dirección explicando sus propios resultados | Por qué subió o bajó una magnitud |
| **7A** · Market Risk | Exposición a tipos, divisa y precios | Cuantitativo y corto |
| **8** · Financial Statements | Los estados financieros y sus notas | Cifras, y de dónde salen |

Dos cosas que hay que saber del corpus antes de tocarlo:

**`fiscal_year` no es el año de presentación.** Las seis compañías cierran
ejercicio en cuatro meses distintos —NVDA en enero, MSFT en junio, AAPL en
septiembre, y GOOGL, META y AMZN en diciembre—, y está elegido así a
propósito. El 10-K de NVDA FY2025 se presentó en febrero de 2025; el de
Alphabet FY2025, en febrero de **2026**. Quien razone por fecha de
presentación se equivoca.

**NVIDIA no pone sus estados financieros bajo el Item 8.** Los deja bajo el
Item 15 y en el 8 escribe una remisión de dos líneas. El corpus sirve el
contenido correcto bajo la clave `"8"` y deja constancia en el campo
`item_origen`. Si no lo hiciera, `read_section("NVDA", 2025, "8")` devolvería
cuarenta tokens inútiles.


In [ ]:
# Cuánto ocupa un 10-K. Los tokens vienen precalculados en el corpus: contar
# en vivo tardaría más que la clase entera.
import json
import pandas as pd

secciones = pd.DataFrame(
    json.loads(l) for l in open("corpus/secciones.jsonl", encoding="utf-8")
)

tabla = secciones.pivot_table(
    index=["ticker", "fiscal_year"], columns="item", values="n_tokens"
).astype(int)
tabla["TOTAL"] = tabla.sum(axis=1)
print(tabla.to_string())

print(f"\nCorpus entero: {secciones.n_tokens.sum():,} tokens en "
      f"{len(secciones)} secciones")
print(f"Informe medio: {tabla['TOTAL'].mean():,.0f} tokens")
print(f"Informe mayor: {tabla['TOTAL'].max():,} · menor: "
      f"{tabla['TOTAL'].min():,}")

mayor = secciones.nlargest(1, "n_tokens").iloc[0]
# Ojo con `mayor.item`: en pandas eso es el método Series.item, no la
# columna. Con una columna que se llama 'item' hay que usar corchetes.
print(f"Sección mayor: {mayor['ticker']} FY{mayor['fiscal_year']} "
      f"Item {mayor['item']} con {mayor['n_tokens']:,} tokens")


## Qué es una herramienta, y qué ve el modelo de ella

Una *tool* es una función de Python que el modelo puede pedir que se ejecute.
El decorador `@tool` la convierte en un esquema —nombre, parámetros con sus
tipos, y descripción— y ese esquema viaja en la petición junto a los mensajes.

Lo que hay que entender es **qué parte de vuestra función ve el modelo**:

| Lo ve | No lo ve |
| --- | --- |
| El nombre de la función | El cuerpo |
| Los nombres y tipos de los parámetros | Los comentarios |
| El *docstring*, entero | Cómo de rápida o cara es |
| Lo que devuelve, cuando la llama | Lo que hace por dentro |

De ahí sale la consecuencia que gobierna el resto de la sesión: **el modelo
decide si os llama leyendo el docstring**. Un docstring vago produce un agente
que elige mal, con el mismo código debajo. Volveremos a esto en el segundo
ejercicio.

Empezamos por la herramienta fácil: exacta, determinista y prácticamente
gratis.


In [ ]:
from langchain.tools import tool

xbrl = pd.read_parquet("corpus/xbrl_facts.parquet")
print(f"{len(xbrl)} hechos XBRL · {xbrl.concept.nunique()} conceptos "
      f"distintos · {xbrl.ticker.nunique()} compañías")


@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve el valor EXACTO de una magnitud financiera tal y como la
    compañía la reportó en XBRL.

    Es la fuente autorizada para cualquier cifra. Úsala SIEMPRE en lugar de
    leer un número del texto del informe.

    Args:
        ticker: Símbolo bursátil, p. ej. 'NVDA'.
        fiscal_year: Ejercicio fiscal reportado, p. ej. 2024.
        concept: Concepto US-GAAP, p. ej. 'Revenues', 'NetIncomeLoss',
            'Assets', 'OperatingIncomeLoss'.

    Devuelve el valor con su unidad y fecha de cierre, o un aviso explícito
    si la compañía no reportó ese concepto en ese ejercicio.
    """
    filas = xbrl[
        (xbrl.ticker == ticker)
        & (xbrl.fiscal_year == int(fiscal_year))
        & (xbrl.concept == concept)
    ]

    if filas.empty:
        disponibles = sorted(
            xbrl[
                (xbrl.ticker == ticker)
                & (xbrl.fiscal_year == int(fiscal_year))
            ].concept.unique()
        )
        if not disponibles:
            return (
                f"No hay datos de {ticker} para FY{fiscal_year} en el corpus. "
                "Usa list_available para ver qué hay."
            )
        return (
            f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
            f"Conceptos disponibles: {', '.join(disponibles)}"
        )

    f = filas.iloc[0]
    return (
        f"{ticker} FY{fiscal_year} · {concept} = {f.value:,.0f} {f.unit} "
        f"(cierre de ejercicio {f.period_end}, según el {f.form})"
    )


In [ ]:
import miax_s1


@tool
def search_filings(
    query: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> str:
    """Busca fragmentos de texto relevantes en los informes 10-K del corpus.

    Úsala para preguntas cualitativas: riesgos, estrategia, litigios y
    comentarios de la dirección. NO la uses para obtener cifras: para eso
    está get_xbrl_fact.

    Args:
        query: Consulta de búsqueda, preferiblemente en inglés.
        ticker: Filtra por compañía si la pregunta la menciona.
        fiscal_year: Filtra por ejercicio si la pregunta lo menciona.
        item: '1A', '7', '7A' u '8'.
        k: Número de fragmentos a devolver.

    Devuelve fragmentos con chunk_id para poder citarlos.
    """
    return miax_s1.formatear_fragmentos(
        miax_s1.buscar(
            query,
            ticker=ticker,
            fiscal_year=fiscal_year,
            item=item,
            k=k,
        )
    )


In [ ]:
@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    """Devuelve el TEXTO COMPLETO de una sección de un 10-K.

    Es una herramienta CARA. Úsala solo cuando search_filings no proporcione
    contexto suficiente y sea necesario leer una sección completa.

    Args:
        ticker: Símbolo bursátil.
        fiscal_year: Ejercicio fiscal.
        item: '1A', '7', '7A' u '8'.
    """
    filas = secciones[
        (secciones.ticker == ticker)
        & (secciones.fiscal_year == int(fiscal_year))
        & (secciones.item == item)
    ]

    if filas.empty:
        return (
            f"No hay Item {item} de {ticker} FY{fiscal_year} en el corpus. "
            "Usa list_available para ver qué hay."
        )

    return filas.iloc[0].texto


In [ ]:
@tool
def list_available() -> str:
    """Lista qué compañías, ejercicios y secciones existen en el corpus.

    Úsala antes de responder que un dato no existe y cuando no estés seguro
    de que la compañía o el ejercicio solicitado estén disponibles.
    """
    orden_items = {"1A": 0, "7": 1, "7A": 2, "8": 3}
    lineas = []

    for ticker, filas in secciones.groupby("ticker", sort=True):
        empresas = ", ".join(
            sorted(filas["empresa"].dropna().astype(str).unique())
        )
        ejercicios = ", ".join(
            f"FY{anio}"
            for anio in sorted(
                filas["fiscal_year"].dropna().astype(int).unique()
            )
        )
        items = sorted(
            filas["item"].dropna().astype(str).unique(),
            key=lambda item: (orden_items.get(item, len(orden_items)), item),
        )
        lineas.append(
            f"{ticker} — {empresas}: {ejercicios} · Items: {', '.join(items)}"
        )

    return "\n".join(lineas)


HERRAMIENTAS = [
    list_available,
    get_xbrl_fact,
    search_filings,
    read_section,
]
NOMBRES_HERRAMIENTAS = {tool.name for tool in HERRAMIENTAS}

print("Herramientas:", ", ".join(t.name for t in HERRAMIENTAS))


## Agente baseline

El baseline conserva las cuatro firmas de herramientas del enunciado, usa
salida estructurada y limita el número de llamadas por invocación.

`ToolCallLimitMiddleware` también cuenta la llamada usada por LangChain para
materializar `RespuestaFinanciera`. Por eso se usa `run_limit=11`: deja margen
para hasta 10 llamadas operativas y una llamada de salida estructurada.


In [ ]:
SYSTEM = """Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

Reglas:
- Para cualquier CIFRA, usa get_xbrl_fact. Nunca leas un número de la prosa.
- Para riesgos, estrategia o comentarios de la dirección, usa search_filings.
- Si no sabes si una compañía o un ejercicio están en el corpus, empieza por
  list_available.
- El corpus está en inglés: escribe las consultas de búsqueda en inglés.
- Cita el chunk_id del fragmento en el que te apoyes.
- Si el dato no está en el corpus, dilo. No lo estimes.
"""


In [ ]:
# El esquema de respuesta. Es el contrato §7 del enunciado, literal.
#
# Fijaos en lo que hace: convierte la cita de una súplica en el prompt
# ("por favor, cita la fuente") en un requisito estructural. El modelo no
# puede devolver una respuesta sin decir de dónde sale, porque el esquema no
# valida. Y los evaluadores del día 17 leen campos en lugar de parsear prosa.
from typing import Literal

from pydantic import BaseModel, Field


class RespuestaFinanciera(BaseModel):
    """Respuesta trazable a una pregunta sobre informes 10-K."""

    respuesta: str = Field(
        description="Respuesta en prosa, breve y directa")
    cifra: float | None = Field(
        default=None, description="Valor numérico, si la pregunta pide uno")
    unidad: str | None = Field(
        default=None, description="USD, shares, porcentaje…")
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"] = Field(
        description="De dónde sale el dato. 'ninguna' si no está en el corpus")
    cita: str | None = Field(
        default=None,
        description="Texto literal del informe que respalda la respuesta")
    chunk_id: str | None = Field(
        default=None,
        description="Identificador del fragmento citado, para verificar")


print(json.dumps(RespuestaFinanciera.model_json_schema()["properties"],
                 indent=2, ensure_ascii=False)[:600], "...")


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

limite_tools = ToolCallLimitMiddleware(
    run_limit=11,
    exit_behavior="end",
)

agente = None
if HAY_CLAVE:
    agente = create_agent(
        model=MODELO,
        tools=HERRAMIENTAS,
        system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        checkpointer=InMemorySaver(),
        middleware=[limite_tools],
    )
    print(
        "Agente montado con 4 herramientas y límite de "
        "10 llamadas operativas por ejecución."
    )
else:
    print("Sin clave: no se puede montar el agente.")


## La práctica

El enunciado completo lo tenéis en la mano. Aquí queda lo imprescindible.

**Qué se entrega**, en grupos de 3:

| | Peso |
| --- | --- |
| Repositorio con el agente, el golden set y los evaluadores | 30 |
| Presentación de 8 minutos el 24, con 10 preguntas ciegas en directo | 70 |

**El golden set.** Recibís 20 preguntas oficiales y escribís **20 vuestras**,
de las cuales **al menos 6 comparativas**. Tres familias:

| Familia | Qué mide | Campos que se rellenan |
| --- | --- | --- |
| `extractiva` | Retrieval y trazabilidad | `item_esperado`, `ancla_texto` |
| `numerica` | El guardrail contra XBRL | `cifra_esperada`, `unidad`, `concept_xbrl` |
| `comparativa` | Que el agente descomponga y compare | los tres, por cada ejercicio |

**Por qué el mínimo de 6 comparativas.** «¿Qué riesgos nuevos añadió entre
FY2024 y FY2025?» no la contesta una sola recuperación: hay que descomponer,
recuperar dos veces y comparar. Es la familia donde el agente deja de ser
decoración sobre una *pipeline*, y sin ella vuestro informe no puede demostrar
que hiciera falta un agente. Hay sustancia real que encontrar: el Item 1A
cambia entre el 49 % y el 85 % de los párrafos entre los dos ejercicios, según
la compañía.

**La verdad se ancla a una frase, no a un `chunk_id`.** El día 17 vais a
cambiar el troceado, y en cuanto lo toquéis todos los `chunk_id` son otros. Si
la métrica dependiera de ellos, el grupo que mejorase el troceado saldría
penalizado por haberlo mejorado. Por eso `ancla_texto` es un texto literal del
informe: **una frase**, no tres párrafos.


In [ ]:
# El golden set oficial: las 20 preguntas con respuesta conocida.
from pathlib import Path

RUTA_GOLDEN = Path("golden_set.jsonl")
if not RUTA_GOLDEN.is_file():
    RUTA_GOLDEN = Path("golden_set_ejemplo.jsonl")   # provisional

golden = [json.loads(l) for l in open(RUTA_GOLDEN, encoding="utf-8")
          if l.strip()]
g = pd.DataFrame(golden)

print(f"{RUTA_GOLDEN.name}: {len(g)} preguntas")
print("\nreparto por familia:")
print(g.familia.value_counts().to_string())
print("\nherramienta esperada:")
print(g.herramienta_esperada.explode().value_counts().to_string())

for fila in golden[:2]:
    print("\n" + json.dumps(fila, ensure_ascii=False, indent=2))

# NOTA PARA EL 10 DE SEPTIEMBRE: si arriba pone `golden_set_ejemplo.jsonl`,
# es que las 20 oficiales todavía no están en la carpeta. El fichero de
# ejemplo tiene tres preguntas y sirve solo para ver el esquema y probar el
# validador de la celda siguiente.


In [ ]:
# Vuestras 20 preguntas, y el validador que tienen que pasar antes de
# entregarlas. Un golden set que no pasa el validador no se corrige: se
# devuelve.
PLANTILLA = {
    "id": "g3-001",
    "pregunta": "¿Cuál fue el revenue de NVIDIA en el ejercicio 2024?",
    "familia": "numerica",              # extractiva | numerica | comparativa
    "ticker": "NVDA",
    "fiscal_year": 2024,
    "respuesta_esperada": "60.922 millones de dólares",
    "cifra_esperada": 60922000000.0,
    "unidad": "USD",
    "concept_xbrl": "Revenues",
    "item_esperado": None,
    "ancla_texto": None,
    "ancla_inicio": None,
    "ancla_fin": None,
    "chunk_id_esperado": None,
    "herramienta_esperada": ["get_xbrl_fact"],
    "autor": "grupo-3",
}

CAMPOS = set(PLANTILLA)
FAMILIAS = {"extractiva", "numerica", "comparativa"}


def validar(preguntas: list[dict], exigir_20: bool = True) -> list[str]:
    """Los problemas del fichero, uno por línea. Lista vacía = correcto."""
    problemas = []
    tickers = set(secciones.ticker)
    ejercicios = set(secciones.fiscal_year.astype(int))
    vistos = set()

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        if faltan := CAMPOS - set(p):
            problemas.append(f"{pid}: faltan campos {sorted(faltan)}")
            continue
        if p["id"] in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(p["id"])
        if p["familia"] not in FAMILIAS:
            problemas.append(f"{pid}: familia '{p['familia']}' no válida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: {p['ticker']} no está en el corpus")
        if int(p["fiscal_year"]) not in ejercicios:
            problemas.append(f"{pid}: FY{p['fiscal_year']} no está en el "
                             f"corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p.get("cifra_esperada") is None:
                problemas.append(f"{pid}: numérica sin cifra_esperada")
            concepto = p.get("concept_xbrl")
            hay = xbrl[(xbrl.ticker == p["ticker"])
                       & (xbrl.fiscal_year == int(p["fiscal_year"]))
                       & (xbrl.concept == concepto)]
            if concepto and hay.empty:
                problemas.append(
                    f"{pid}: {p['ticker']} no reporta '{concepto}' en "
                    f"FY{p['fiscal_year']}. El concepto se mira en "
                    f"xbrl_facts.parquet, nunca por analogía con otra "
                    f"compañía.")
        if p["familia"] in {"extractiva", "comparativa"}:
            ancla = p.get("ancla_texto")
            if not ancla:
                problemas.append(f"{pid}: extractiva sin ancla_texto")
            elif len(ancla.split()) > 40:
                problemas.append(
                    f"{pid}: ancla de {len(ancla.split())} palabras. Una "
                    f"frase. Así no medís vuestro retrieval, medís vuestro "
                    f"tamaño de ventana.")
        if not p.get("herramienta_esperada"):
            problemas.append(f"{pid}: sin herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(f"hacen falta 20 preguntas, hay {len(preguntas)}")
        n_comp = sum(p.get("familia") == "comparativa" for p in preguntas)
        if n_comp < 6:
            problemas.append(f"hacen falta 6 comparativas, hay {n_comp}")
    return problemas


problemas = validar(golden, exigir_20=False)
print("Validando el fichero cargado:")
print("\n".join(f"  - {p}" for p in problemas) or "  sin problemas")

# --- verificación de §7 --------------------------------------------------
assert not validar([PLANTILLA], exigir_20=False), \
    "La plantilla debería pasar su propio validador."
assert validar([{**PLANTILLA, "ticker": "TSLA"}], exigir_20=False), \
    "El validador tiene que rechazar una compañía que no está en el corpus."
print("\n§7 listo. El validador funciona; ahora escribid las preguntas.")


## Instrumentación del baseline

Estas funciones sustituyen todas las celdas de prueba ad hoc que se habían
ido acumulando. No ejecutan preguntas por sí solas.

Se registran:
- respuesta estructurada;
- trayectoria y salida de cada herramienta;
- número de llamadas operativas;
- tokens de todas las llamadas al modelo;
- latencia total por pregunta.


In [ ]:
import time
import uuid


def responder(pregunta: str):
    """Ejecuta una pregunta aislada contra el agente baseline."""
    if agente is None:
        raise RuntimeError("El agente no está disponible.")

    config = {
        "configurable": {
            "thread_id": f"eval-{uuid.uuid4()}"
        }
    }

    return agente.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config=config,
    )


def llamadas_operativas(resultado):
    """Nombres de las cuatro herramientas operativas realmente utilizadas."""
    llamadas = []

    for mensaje in resultado["messages"]:
        for call in getattr(mensaje, "tool_calls", []) or []:
            nombre = call.get("name")
            if nombre in NOMBRES_HERRAMIENTAS:
                llamadas.append(nombre)

    return llamadas


def uso_tokens(resultado):
    """Suma el uso de todas las llamadas al modelo de una ejecución."""
    totales = {
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
        "reasoning_tokens": 0,
        "model_calls": 0,
    }

    for mensaje in resultado["messages"]:
        usage = getattr(mensaje, "usage_metadata", None)
        if not usage:
            continue

        totales["model_calls"] += 1
        totales["input_tokens"] += usage.get("input_tokens", 0)
        totales["output_tokens"] += usage.get("output_tokens", 0)
        totales["total_tokens"] += usage.get("total_tokens", 0)

        detalles = usage.get("output_token_details") or {}
        totales["reasoning_tokens"] += detalles.get("reasoning", 0)

    return totales


def trayectoria_operativa(resultado):
    """Devuelve llamadas operativas, argumentos y salida de cada herramienta."""
    salidas = {}

    for mensaje in resultado["messages"]:
        tool_call_id = getattr(mensaje, "tool_call_id", None)
        if tool_call_id:
            contenido = getattr(mensaje, "content", None)
            salidas[tool_call_id] = (
                contenido if isinstance(contenido, str)
                else str(contenido)
            )

    trayectoria = []

    for mensaje in resultado["messages"]:
        for call in getattr(mensaje, "tool_calls", []) or []:
            nombre = call.get("name")
            if nombre not in NOMBRES_HERRAMIENTAS:
                continue

            call_id = call.get("id")
            trayectoria.append({
                "id": call_id,
                "name": nombre,
                "args": call.get("args", {}),
                "output": salidas.get(call_id),
            })

    return trayectoria


def registro_resultado(resultado, latencia_s):
    """Convierte una ejecución del agente en un registro serializable."""
    respuesta = resultado.get("structured_response")

    if hasattr(respuesta, "model_dump"):
        respuesta = respuesta.model_dump()

    return {
        "respuesta": respuesta,
        "trayectoria": trayectoria_operativa(resultado),
        "n_tool_calls": len(llamadas_operativas(resultado)),
        **uso_tokens(resultado),
        "latencia_s": latencia_s,
    }


## Evaluadores automáticos

Se mantienen separados:
1. evaluación numérica;
2. evaluación de cita/retrieval;
3. evaluación de trayectoria.

Una llamada adicional no convierte por sí sola una trayectoria correcta en
incorrecta: se registra como **ineficiencia**. Esto permite distinguir routing
incorrecto de trabajo innecesario.


In [ ]:
import math
import re


def _valor_xbrl(ticker, fiscal_year, concept):
    filas = xbrl[
        (xbrl["ticker"] == ticker)
        & (xbrl["fiscal_year"] == int(fiscal_year))
        & (xbrl["concept"] == concept)
    ]

    if len(filas) != 1:
        raise ValueError(
            f"Se esperaba un único hecho XBRL para "
            f"{ticker} FY{fiscal_year} {concept}; encontrados: {len(filas)}"
        )

    return float(filas.iloc[0]["value"])


def evaluar_numerico(registro):
    """Evalúa el campo numérico estructurado frente al ground truth XBRL."""
    esperado = registro["esperado"]
    observado = registro.get("respuesta") or {}

    cifra_esperada = esperado.get("cifra")

    if cifra_esperada is None:
        return {
            "aplicable": False,
            "ok": None,
            "cifra_ok": None,
            "unidad_ok": None,
        }

    cifra_observada = observado.get("cifra")
    unidad_esperada = esperado.get("unidad")
    unidad_observada = observado.get("unidad")

    unidad_ok = (
        unidad_esperada is None
        or (
            unidad_observada is not None
            and str(unidad_observada).upper()
            == str(unidad_esperada).upper()
        )
    )

    if registro["familia"] == "numerica":
        cifra_ok = (
            cifra_observada is not None
            and math.isclose(
                float(cifra_observada),
                float(cifra_esperada),
                rel_tol=1e-9,
                abs_tol=0.01,
            )
        )

        return {
            "aplicable": True,
            "ok": cifra_ok and unidad_ok,
            "cifra_ok": cifra_ok,
            "unidad_ok": unidad_ok,
            "cifra_esperada": cifra_esperada,
            "cifra_observada": cifra_observada,
            "interpretacion": "valor",
        }

    # Comparativas: el esquema permite una sola cifra, aunque la respuesta
    # puede contener varias. Se acepta que represente el valor final o el
    # cambio absoluto entre FY anterior y FY actual.
    ticker = esperado["ticker"]
    fy_actual = int(esperado["fiscal_year"])
    fy_anterior = fy_actual - 1
    concepto = esperado["concept_xbrl"]

    valor_anterior = _valor_xbrl(ticker, fy_anterior, concepto)
    valor_actual = _valor_xbrl(ticker, fy_actual, concepto)
    cambio_absoluto = valor_actual - valor_anterior

    valor_final_ok = (
        cifra_observada is not None
        and math.isclose(
            float(cifra_observada),
            valor_actual,
            rel_tol=1e-9,
            abs_tol=0.01,
        )
    )

    cambio_ok = (
        cifra_observada is not None
        and math.isclose(
            float(cifra_observada),
            cambio_absoluto,
            rel_tol=1e-9,
            abs_tol=0.01,
        )
    )

    cifra_ok = valor_final_ok or cambio_ok

    if valor_final_ok:
        interpretacion = "valor_final"
    elif cambio_ok:
        interpretacion = "cambio_absoluto"
    else:
        interpretacion = None

    return {
        "aplicable": True,
        "ok": cifra_ok and unidad_ok,
        "cifra_ok": cifra_ok,
        "unidad_ok": unidad_ok,
        "cifra_observada": cifra_observada,
        "valor_anterior_xbrl": valor_anterior,
        "valor_actual_xbrl": valor_actual,
        "cambio_absoluto_xbrl": cambio_absoluto,
        "interpretacion": interpretacion,
    }


def chunks_recuperados(registro):
    """Extrae {chunk_id: texto} de las llamadas a search_filings."""
    chunks = {}

    for paso in registro.get("trayectoria", []):
        if paso["name"] != "search_filings":
            continue

        output = paso.get("output") or ""
        patron = r"(?ms)^\[([^\]]+)\].*?(?=^\[[^\]]+\]|\Z)"

        for match in re.finditer(patron, output):
            chunk_id = match.group(1).strip()
            chunks[chunk_id] = match.group(0)

    return chunks


def evaluar_cita(registro):
    """Evalúa si el retrieval recuperó el ancla y si la respuesta la citó."""
    ancla = registro["esperado"].get("ancla_texto")

    if not ancla:
        return {
            "aplicable": False,
            "retrieval_hit": None,
            "cita_ok": None,
        }

    chunks = chunks_recuperados(registro)

    chunks_con_ancla = [
        chunk_id
        for chunk_id, texto in chunks.items()
        if ancla in texto
    ]

    retrieval_hit = bool(chunks_con_ancla)
    respuesta = registro.get("respuesta") or {}

    cita_observada = respuesta.get("chunk_id")
    if not cita_observada:
        cita_observada = respuesta.get("cita")

    cita_ok = bool(
        cita_observada
        and any(
            chunk_id in str(cita_observada)
            for chunk_id in chunks_con_ancla
        )
    )

    return {
        "aplicable": True,
        "retrieval_hit": retrieval_hit,
        "cita_ok": cita_ok,
        "ancla_encontrada_en": chunks_con_ancla,
        "chunk_esperado": registro["esperado"].get("chunk_id"),
        "cita_observada": cita_observada,
    }


def _entero_seguro(valor):
    try:
        return int(valor)
    except (TypeError, ValueError):
        return None


def evaluar_trayectoria(registro):
    """Evalúa routing/cobertura y separa corrección de eficiencia."""
    esperado = registro["esperado"]
    familia = registro["familia"]
    trayectoria = registro.get("trayectoria", [])

    esperadas = set(esperado["herramientas"])
    auxiliares = {"list_available"}

    observadas_lista = [p["name"] for p in trayectoria]
    observadas = set(observadas_lista)

    faltantes = sorted(esperadas - observadas)
    inesperadas = sorted(observadas - esperadas - auxiliares)

    llamadas_extra = []

    # Herramientas operativas no requeridas: ineficiencia, no fallo de routing.
    for paso in trayectoria:
        if paso["name"] in inesperadas:
            llamadas_extra.append({
                "name": paso["name"],
                "args": paso.get("args", {}),
                "motivo": "herramienta operativa no necesaria para este caso",
            })

    ticker = esperado["ticker"]
    fy_actual = int(esperado["fiscal_year"])
    concepto = esperado.get("concept_xbrl")
    item = esperado.get("item")

    if familia == "comparativa":
        years_esperados = {fy_actual - 1, fy_actual}
    else:
        years_esperados = {fy_actual}

    diagnostico = {}

    # XBRL ---------------------------------------------------------------
    if "get_xbrl_fact" in esperadas:
        llamadas_xbrl = [
            p for p in trayectoria
            if p["name"] == "get_xbrl_fact"
        ]
        years_xbrl_correctos = set()

        for paso in llamadas_xbrl:
            args = paso.get("args", {})
            fy = _entero_seguro(args.get("fiscal_year"))

            llamada_correcta = (
                args.get("ticker") == ticker
                and fy in years_esperados
                and args.get("concept") == concepto
            )

            if llamada_correcta:
                years_xbrl_correctos.add(fy)
            else:
                llamadas_extra.append({
                    "name": paso["name"],
                    "args": args,
                    "motivo": "argumentos XBRL no esperados",
                })

        cobertura_xbrl_ok = (
            years_xbrl_correctos == years_esperados
        )

        diagnostico["xbrl"] = {
            "years_esperados": sorted(years_esperados),
            "years_correctos": sorted(years_xbrl_correctos),
            "cobertura_ok": cobertura_xbrl_ok,
        }
    else:
        cobertura_xbrl_ok = True

    # Retrieval textual --------------------------------------------------
    if "search_filings" in esperadas:
        llamadas_search = [
            p for p in trayectoria
            if p["name"] == "search_filings"
        ]
        years_search_correctos = set()

        for paso in llamadas_search:
            args = paso.get("args", {})
            fy = _entero_seguro(args.get("fiscal_year"))

            llamada_correcta = (
                args.get("ticker") == ticker
                and fy in years_esperados
                and (
                    item is None
                    or str(args.get("item")) == str(item)
                )
            )

            if llamada_correcta:
                years_search_correctos.add(fy)
            else:
                llamadas_extra.append({
                    "name": paso["name"],
                    "args": args,
                    "motivo": "filtros de búsqueda no esperados",
                })

        cobertura_search_ok = (
            years_search_correctos == years_esperados
        )

        diagnostico["search"] = {
            "years_esperados": sorted(years_esperados),
            "years_correctos": sorted(years_search_correctos),
            "cobertura_ok": cobertura_search_ok,
        }
    else:
        cobertura_search_ok = True

    ok = (
        not faltantes
        and cobertura_xbrl_ok
        and cobertura_search_ok
    )

    return {
        "ok": ok,
        "eficiente": len(llamadas_extra) == 0,
        "esperadas": sorted(esperadas),
        "observadas": observadas_lista,
        "faltantes": faltantes,
        "inesperadas": inesperadas,
        "diagnostico": diagnostico,
        "n_llamadas_extra": len(llamadas_extra),
        "llamadas_extra": llamadas_extra,
    }


In [ ]:
def evaluar_una(caso):
    """Ejecuta un caso con ground truth y añade los tres evaluadores."""
    inicio = time.perf_counter()
    resultado = responder(caso["pregunta"])
    latencia_s = time.perf_counter() - inicio

    registro = {
        "id": caso["id"],
        "familia": caso["familia"],
        "pregunta": caso["pregunta"],
        "esperado": {
            "ticker": caso["ticker"],
            "fiscal_year": caso["fiscal_year"],
            "respuesta": caso["respuesta_esperada"],
            "cifra": caso["cifra_esperada"],
            "unidad": caso["unidad"],
            "concept_xbrl": caso["concept_xbrl"],
            "item": caso["item_esperado"],
            "ancla_texto": caso["ancla_texto"],
            "ancla_inicio": caso["ancla_inicio"],
            "ancla_fin": caso["ancla_fin"],
            "chunk_id": caso["chunk_id_esperado"],
            "herramientas": caso["herramienta_esperada"],
        },
        **registro_resultado(resultado, latencia_s),
    }

    registro["evaluacion"] = {
        "numerico": evaluar_numerico(registro),
        "cita": evaluar_cita(registro),
        "trayectoria": evaluar_trayectoria(registro),
    }

    return registro


def evaluar(ruta_jsonl):
    """Ejecuta todos los casos de un JSONL y devuelve registros + resumen."""
    ruta = Path(ruta_jsonl)
    casos = [
        json.loads(linea)
        for linea in ruta.read_text(encoding="utf-8").splitlines()
        if linea.strip()
    ]

    if not casos:
        raise ValueError(f"No hay casos en {ruta}")

    # Para el golden propio se puede validar el esquema sin exigir 20,
    # de modo que la misma función sirva también para un holdout de 10.
    problemas = validar(casos, exigir_20=False)
    if problemas:
        raise ValueError(
            "El JSONL no pasa el validador:\n- " + "\n- ".join(problemas)
        )

    registros = [evaluar_una(caso) for caso in casos]

    filas = []
    for r in registros:
        ev_num = r["evaluacion"]["numerico"]
        ev_cita = r["evaluacion"]["cita"]
        ev_tray = r["evaluacion"]["trayectoria"]

        if r["familia"] == "numerica":
            acierto = bool(ev_num["ok"] and ev_tray["ok"])
        elif r["familia"] == "extractiva":
            acierto = bool(
                ev_cita["retrieval_hit"]
                and ev_cita["cita_ok"]
                and ev_tray["ok"]
            )
        else:  # comparativa
            acierto = bool(
                ev_num["ok"]
                and ev_cita["retrieval_hit"]
                and ev_cita["cita_ok"]
                and ev_tray["ok"]
            )

        filas.append({
            "id": r["id"],
            "familia": r["familia"],
            "acierto": acierto,
            "retrieval_hit": ev_cita.get("retrieval_hit"),
            "cita_ok": ev_cita.get("cita_ok"),
            "numerico_ok": ev_num.get("ok"),
            "trayectoria_ok": ev_tray.get("ok"),
            "trayectoria_eficiente": ev_tray.get("eficiente"),
            "n_tool_calls": r["n_tool_calls"],
            "model_calls": r["model_calls"],
            "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"],
            "total_tokens": r["total_tokens"],
            "latencia_s": r["latencia_s"],
        })

    resumen = pd.DataFrame(filas)
    return registros, resumen


### Estado de este notebook limpio

Se han eliminado las celdas de depuración y pruebas puntuales (`N1`, `E1`,
`C5`, inspección del middleware, impresiones de trazas, etc.).

Antes de congelar el baseline todavía hay dos piezas que deben tratarse de
forma explícita en la práctica:

- el **middleware/guardrail numérico** que devuelve al modelo un desajuste
  frente a XBRL para que corrija la respuesta;
- el cálculo de **coste monetario** con la tarifa verificada del modelo usado.

No se han inventado esas dos piezas dentro de esta limpieza.
